# How to Build Skills

A **skill** is a reusable, self-contained instruction set that tells an AI agent how to handle a specific category of task. Skills live at `~/.claude/commands/` and are invoked by name. They are the atomic building blocks of an AI-native workflow.

## The agentskills.io Specification

Skills follow the [agentskills.io](https://agentskills.io) spec. Each skill is a folder containing:

```
my-skill/
├── SKILL.md          # Required: the skill definition and instructions
├── scripts/          # Optional: executable tools the skill calls
├── references/       # Optional: static reference data the skill reads
└── assets/           # Optional: templates, examples, other static files
```

The `SKILL.md` file is the heart of every skill. Everything else is support.

## SKILL.md Anatomy

```markdown
---
name: my-skill
description: One sentence explaining when this skill activates and what it does.
allowed-tools: Read, Bash, Edit
triggers:
  - "phrase that invokes this"
  - "another trigger phrase"
---

# Skill Title

Instructions for the agent. Written as imperative prose.
```

### Frontmatter fields

| Field | Required | Purpose |
|-------|----------|---------|
| `name` | Yes | Unique identifier. Used for invocation and file naming. |
| `description` | Yes | Loaded into every session. Must be specific enough for the agent to know when to use this skill. |
| `allowed-tools` | No | Restricts which tools the agent can use while executing this skill. |
| `triggers` | No | Phrases that should automatically invoke this skill. |

## The Tool-Use Pattern

The most powerful skills follow the **tool-use pattern**: instead of one monolithic script, each operation is a separate atomic tool with a defined JSON input and output schema.

### Why atomic tools?

- The agent can call them selectively based on context
- Each tool is independently testable
- Outputs compose cleanly: tool A's output is tool B's input
- You avoid giant scripts that try to do everything and do nothing well

### Tool contract

Every tool in `scripts/` should:

1. Accept input via environment variables or positional args
2. Emit a single JSON object to stdout
3. Emit human-readable diagnostics to stderr (not stdout)
4. Exit 0 on success, non-zero on failure

```bash
# Good: one purpose, JSON out, errors to stderr
result=$(bash scripts/list.sh)
echo "$result" | jq '.applications[].name'

# Bad: giant script with subcommands, mixed output
bash manage.sh --list --format=human
```

## Tool Schema Documentation

Document every tool inside SKILL.md using this pattern:

```markdown
## Tools

### scripts/list.sh
Lists all deployable applications.
- Input: none
- Output: `{applications: [{uuid, name, status, fqdn}], services: [...]}`

### scripts/search.sh
Fuzzy-matches an application by name.
- Input: `QUERY=<string>` env var
- Output: `{matched: bool, uuid, name, candidates: [{uuid, name, status}]}`
- `matched: true` means exactly one hit. `matched: false` means show candidates.

### scripts/deploy.sh
Triggers a deploy for a specific application.
- Input: `<uuid>`
- Output: `{triggered: bool, uuid, name}`
```

The agent reads this schema and knows which tool to call without having to guess.

## Writing a Tool Script

Here is a complete, production-quality tool script:

In [ ]:
# Example: scripts/list.sh
# Lists items from an API, returns clean JSON

example_script = '''
#!/usr/bin/env bash
set -euo pipefail

# Read secrets from a known location, never hardcode
SECRETS="${HOME}/.secrets/master.env"
API_KEY=$(grep -m1 '^MY_API_KEY=' "$SECRETS" | cut -d= -f2-)
BASE_URL=$(grep -m1 '^MY_BASE_URL=' "$SECRETS" | cut -d= -f2-)

# Write API response to a temp file, not a variable
# (variable interpolation breaks on JSON with quotes)
tmp=$(mktemp)
trap 'rm -f "$tmp"' EXIT

curl -sf -H "Authorization: Bearer ${API_KEY}" \\
     "${BASE_URL}/api/v1/items" > "$tmp"

# Transform with Python for reliability
python3 - "$tmp" <<'EOF'
import json, sys

with open(sys.argv[1]) as f:
    data = json.load(f)

items = [
    {"id": item["id"], "name": item["name"], "status": item.get("status", "unknown")}
    for item in data
]

print(json.dumps({"items": items, "count": len(items)}, indent=2))
EOF
'''

print(example_script)

## Scan-Before-Act Pattern

For any skill that modifies state, always scan first:

1. `scan.sh` -- surveys the environment with no side effects, returns what exists
2. Agent synthesizes: "I found X, Y, Z. Here is what I can clean."
3. `clean.sh` -- executes based on the scan result

Never ask the user "should I include VPS?" when you can detect whether VPS credentials exist. Read the environment, decide, act.

```markdown
## Workflow

1. Always run `scripts/scan.sh` first. Never skip.
2. Synthesize: report what you found and what action you will take.
3. Run the appropriate clean/deploy/modify script.
4. Report: before vs. after, concisely.
```

## Installing a Skill

A skill must be placed in `~/.claude/commands/<skill-name>/` to be available.

```bash
# From the skills registry
cp -r ~/dev/mad-house/skills/skills/my-skill ~/.claude/commands/my-skill

# Or use the skill-publish skill to handle this automatically
# /skill-publish my-skill
```

After installing, the skill's `description` field is immediately visible to the agent. No restart required.

### The skills registry

Mad House maintains a registry at `~/dev/mad-house/skills/`. The public version lives at `github.com/madebymadhouse/skills`.

Internal skills (those using secrets, VPS addresses, or internal tooling) stay in the local registry and are never pushed to the public repo.

## Writing Your First Skill: Step by Step

We will build a skill called `git-summary` that reports the state of all git repos under `~/dev`.

In [ ]:
import os

# Step 1: Create the skill folder
skill_path = os.path.expanduser("~/.claude/commands/git-summary")
scripts_path = os.path.join(skill_path, "scripts")

os.makedirs(scripts_path, exist_ok=True)
print(f"Created: {skill_path}")
print(f"Created: {scripts_path}")

In [ ]:
skill_md = '''\
---
name: git-summary
description: Report the git state of all repos under ~/dev. Use when the user asks
  "what repos are dirty", "anything uncommitted", or "git status everywhere".
allowed-tools: Bash
---

# Git Summary

## Tools

### scripts/scan.sh
Scans all repos under ~/dev for dirty state and unpushed commits.
- Input: none
- Output: `{repos: [{name, path, dirty: bool, ahead: int, behind: int}]}`

## Workflow

1. Run `scripts/scan.sh`.
2. Report: list dirty repos and their file counts. List repos with unpushed commits.
3. If everything is clean, say so in one line.
'''

skill_file = os.path.join(skill_path, "SKILL.md")
with open(skill_file, "w") as f:
    f.write(skill_md)

print(f"Written: {skill_file}")

In [ ]:
scan_sh = '''\
#!/usr/bin/env bash
set -euo pipefail

results=[]

python3 - <<'PYEOF'
import subprocess, json, os
from pathlib import Path

dev = Path(os.path.expanduser("~/dev"))
repos = []

for gitdir in sorted(dev.glob("**/.git")):
    if gitdir.parts.count(".git") > 1:
        continue  # skip nested .git dirs
    repo = gitdir.parent
    name = str(repo.relative_to(dev))

    status = subprocess.run(
        ["git", "-C", str(repo), "status", "--porcelain"],
        capture_output=True, text=True
    ).stdout.strip()

    ahead = subprocess.run(
        ["git", "-C", str(repo), "rev-list", "@{u}..", "--count"],
        capture_output=True, text=True
    ).stdout.strip()

    repos.append({
        "name": name,
        "path": str(repo),
        "dirty": bool(status),
        "dirty_count": len(status.splitlines()) if status else 0,
        "ahead": int(ahead) if ahead.isdigit() else 0
    })

print(json.dumps({"repos": repos}, indent=2))
PYEOF
'''

scan_file = os.path.join(scripts_path, "scan.sh")
with open(scan_file, "w") as f:
    f.write(scan_sh)

os.chmod(scan_file, 0o755)
print(f"Written: {scan_file}")

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Interpolating JSON into a heredoc string | Write API responses to temp files with `mktemp` |
| Giant monolithic script with subcommands | One script per operation |
| Asking the user questions the env can answer | Read the environment, detect, decide |
| Hardcoding secrets | Read from `~/.secrets/master.env` |
| Mixing human output with JSON on stdout | Errors and diagnostics go to stderr |
| rsync with `--delete` when staging to a repo | `--delete` wipes repo-only files. Never use it. |

## What makes a great skill description?

The description is always loaded, even when the skill is not active. It must be specific enough that the agent knows exactly when to use it.

**Weak:** `Manages deployments.`

**Strong:** `Deploy, restart, or check status of any Coolify application. Use when deploying a new version, restarting a service, or reading deployment logs. Triggers on: deploy, restart service, push to production.`